# Pharmagen — 3. The graphs the model reads

This notebook opens the real library and shows, for both towers, exactly what
reaches the network: node and edge tensors, and drawings of them.

Everything is loaded through `GraphCache`, the same object training and
inference use.

In [ ]:
# --- Environment -----------------------------------------------------------
# Colab runs Python 3.13; Pharmagen and Modephere need 3.14. colab_setup lets
# uv provision 3.14 + the locked dependencies and starts a kernel there. Every
# cell below marked %%pgen runs in that 3.14 kernel and keeps its state.
REPO_URL = "https://github.com/Aderfi/Pgen_dev.git"
BRANCH = "modephere-imp"

# The data bundle (scripts/pack_colab_bundle.py, ~190 MB): a Google Drive path,
# or an https URL to a release asset. Ignored when running locally.
BUNDLE = "/content/drive/MyDrive/pharmagen_colab_bundle.zip"
# BUNDLE = "https://github.com/Aderfi/Pgen_dev/releases/download/demo/pharmagen_colab_bundle.zip"

import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    REPO = Path("/content/dev_Pgen")
    if not REPO.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO)],
            check=True,
        )
else:
    REPO, BUNDLE = Path.cwd().parent, None  # notebooks/ -> repository root

sys.path.insert(0, str(REPO / "notebooks"))
import colab_setup

kernel = colab_setup.prepare(REPO, BUNDLE)

## The library, as the model sees it

In [ ]:
%%pgen
import torch

from src.data.cache import GraphCache, GraphDims

cache = GraphCache.from_settings(inference_mode=True)
print(f"{len(cache.drug_index)} drugs indexed by ChEMBL parent ID\n")

dims = GraphDims()
print("Feature widths — the single source of truth for every tower input:")
for name, value in vars(dims).items():
    print(f"  {name:16} {value:5}")

## The drug tower's input

One `DrugData` object per molecule. It carries two graphs at once: the molecular
graph (atoms and bonds) and the pharmacophore graph (3D interaction points).

In [ ]:
%%pgen
drug = cache.get_drug("CHEMBL1771")  # clopidogrel
print(drug, "\n")

print(f"Molecular graph : {drug.x.shape[0]:3} atoms, {drug.edge_index.shape[1]:3} directed bonds")
print(f"                  node features {tuple(drug.x.shape)}, edge features {tuple(drug.edge_attr.shape)}")
print(f"Descriptors     : {tuple(drug.global_feats.shape)} per-molecule")
print(f"ADMET           : {tuple(drug.admet_feats.shape)}")
print(f"Pharmacophore   : {drug.pharm_x.shape[0]:3} points, {drug.pharm_edge_index.shape[1]:3} pairs")
print(f"                  node features {tuple(drug.pharm_x.shape)}, 3D positions {tuple(drug.pharm_pos.shape)}")
print(f"Has 3D          : {bool(drug.has_3d)}   |   conformers available: {cache.n_conformers('CHEMBL1771')}")
print(f"InChIKey        : {drug.inchikey}")

### The molecule, drawn from its graph

In [ ]:
%%pgen
import matplotlib.pyplot as plt
import networkx as nx
from rdkit import Chem
from rdkit.Chem import Draw

import polars as pl

from src.config import get_settings

compounds = pl.read_parquet(get_settings().paths.library / "compounds.parquet")
smiles = compounds.filter(pl.col("chembl_id") == "CHEMBL1771")["working_smiles"].item()
mol = Chem.MolFromSmiles(smiles)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

axes[0].imshow(Draw.MolToImage(mol, size=(420, 380)))
axes[0].set_title("Clopidogrel (standardized structure)")
axes[0].axis("off")

# The same molecule as the tensor the tower receives: atomic number colours the
# nodes, so the graph and the drawing can be read against each other.
graph = nx.Graph()
graph.add_edges_from(drug.edge_index.t().tolist())
colours = [int(z) for z in drug.z]
nx.draw(
    graph,
    pos=nx.kamada_kawai_layout(graph),
    ax=axes[1],
    node_color=colours,
    cmap="viridis",
    node_size=170,
    width=1.2,
    edge_color="#888888",
    with_labels=False,
)
axes[1].set_title(f"Molecular graph: {drug.x.shape[0]} nodes x {drug.x.shape[1]} features")

plt.tight_layout()
plt.show()

### The pharmacophore graph

The second branch of the drug tower. Each node is an interaction feature of one
low-energy conformer — a donor, acceptor, hydrophobe, aromatic ring or charge —
positioned in 3D. This is what makes two structurally different molecules that
bind the same way look similar to the model.

In [ ]:
%%pgen
import numpy as np
from modephere.types.enums import PharmacophoreFamily

families = [f.value for f in PharmacophoreFamily]
n_fam = len(families)

positions = drug.pharm_pos.numpy()
# The first 6 columns of pharm_x are the one-hot PharmacophoreFamily; columns 6..10 carry geometry/spread.
family_idx = drug.pharm_x[:, :n_fam].argmax(dim=1).tolist()
family_names = [families[i] for i in family_idx]

palette = {
    "Donor": "#1f77b4",
    "Acceptor": "#d62728",
    "PosIonizable": "#9467bd",
    "NegIonizable": "#ff7f0e",
    "Aromatic": "#2ca02c",
    "Hydrophobe": "#8c564b",
}
node_colors = [palette[name] for name in family_names]

fig = plt.figure(figsize=(13, 5.5))

ax = fig.add_subplot(1, 2, 1, projection="3d")
for fam_name in sorted(set(family_names), key=families.index):
    mask = [i for i, name in enumerate(family_names) if name == fam_name]
    pts = positions[mask]
    ax.scatter(*pts.T, color=palette[fam_name], s=180, depthshade=False, label=f"{fam_name} ({len(mask)})")
for i, j in drug.pharm_edge_index.t().tolist():
    ax.plot(*zip(positions[i], positions[j]), color="#bbbbbb", linewidth=0.8)
ax.set_title(f"Pharmacophore, 3D ({len(positions)} points)")
ax.set_xlabel("x (Å)")
ax.set_ylabel("y (Å)")
ax.set_zlabel("z (Å)")
ax.legend(loc="upper left", fontsize=8, frameon=True)

ax2 = fig.add_subplot(1, 2, 2)
pharm_graph = nx.Graph()
pharm_graph.add_nodes_from(range(len(positions)))
pharm_graph.add_edges_from(drug.pharm_edge_index.t().tolist())
labels = {i: f"{i}:{name[:3]}" for i, name in enumerate(family_names)}
nx.draw(
    pharm_graph,
    pos=nx.spring_layout(pharm_graph, seed=0),
    ax=ax2,
    node_color=node_colors,
    node_size=520,
    edge_color="#cccccc",
    labels=labels,
    font_size=7,
    font_color="white",
    font_weight="bold",
)
ax2.set_title("Pharmacophore graph (topology + family label)")

plt.tight_layout()
plt.show()

print("Pharmacophore points on clopidogrel (conformer 0):")
for idx, (fam, pos) in enumerate(zip(family_names, positions)):
    print(f"  node {idx:2d}  {fam:14}  xyz=({pos[0]:+6.2f}, {pos[1]:+6.2f}, {pos[2]:+6.2f}) Å")
print(f"\nPairwise distance range: {np.linalg.norm(positions[:, None] - positions[None], axis=-1).max():.1f} Å")

### Interactive 3D pharmacophore & conformer ensemble

Each drug in the library stores up to 10 low-energy 3D conformers (`n_conformers`). During training, `GraphCache` can sample or switch conformers via `cache.with_conformer(idx)` while keeping the 2D molecular graph shared in memory. Below is an interactive 3D view comparing clopidogrel's pharmacophore points across its top 3 conformers.

In [ ]:
%%pgen
import plotly.graph_objects as go

n_conf = min(3, cache.n_conformers("CHEMBL1771"))
conf_drugs = [cache.with_conformer(c).get_drug("CHEMBL1771") for c in range(n_conf)]

print(f"CHEMBL1771 has {cache.n_conformers('CHEMBL1771')} conformers in the library.")
for c, d_conf in enumerate(conf_drugs):
    pos_c = d_conf.pharm_pos.numpy()
    span = np.linalg.norm(pos_c[:, None] - pos_c[None], axis=-1).max()
    rmsd_0 = np.sqrt(((pos_c - conf_drugs[0].pharm_pos.numpy()) ** 2).sum(axis=-1).mean()) if pos_c.shape == conf_drugs[0].pharm_pos.shape else float("nan")
    print(f"  conformer {c}: {d_conf.pharm_x.shape[0]} pharmacophore points, max span = {span:.2f} Å, RMSD vs conf 0 = {rmsd_0:.2f} Å")

fig3d = go.Figure()
symbols = ["circle", "diamond", "square"]

# Edges for conformer 0
pos0 = conf_drugs[0].pharm_pos.numpy()
edge_x, edge_y, edge_z = [], [], []
for u, v in conf_drugs[0].pharm_edge_index.t().tolist():
    if u < v:
        edge_x.extend([pos0[u, 0], pos0[v, 0], None])
        edge_y.extend([pos0[u, 1], pos0[v, 1], None])
        edge_z.extend([pos0[u, 2], pos0[v, 2], None])
fig3d.add_trace(
    go.Scatter3d(
        x=edge_x, y=edge_y, z=edge_z,
        mode="lines",
        line=dict(color="#cccccc", width=2),
        name="Conf 0 edges",
        hoverinfo="skip",
    )
)

for c, d_conf in enumerate(conf_drugs):
    pos_c = d_conf.pharm_pos.numpy()
    fams_c = [families[i] for i in d_conf.pharm_x[:, :n_fam].argmax(dim=1).tolist()]
    fig3d.add_trace(
        go.Scatter3d(
            x=pos_c[:, 0],
            y=pos_c[:, 1],
            z=pos_c[:, 2],
            mode="markers+text" if c == 0 else "markers",
            text=[f"{i}:{f[:3]}" for i, f in enumerate(fams_c)] if c == 0 else None,
            textposition="top center",
            marker=dict(
                size=8 if c == 0 else 5,
                color=[palette[f] for f in fams_c],
                symbol=symbols[c],
                opacity=0.95 if c == 0 else 0.55,
                line=dict(width=1, color="#222222"),
            ),
            customdata=fams_c,
            hovertemplate=f"<b>Conformer {c}</b><br>Node %{{pointNumber}}: %{{customdata}}<br>x=%{{x:.2f}} Å, y=%{{y:.2f}} Å, z=%{{z:.2f}} Å<extra></extra>",
            name=f"Conformer {c} ({len(pos_c)} pts)",
        )
    )

fig3d.update_layout(
    title="Clopidogrel (CHEMBL1771) — 3D Pharmacophore Ensemble (Conformers 0, 1, 2)",
    scene=dict(xaxis_title="x (Å)", yaxis_title="y (Å)", zaxis_title="z (Å)"),
    margin=dict(l=0, r=0, b=0, t=40),
    height=520,
)
from IPython.display import HTML, display
display(HTML(fig3d.to_html(include_plotlyjs="cdn", full_html=False)))

---

## The genotype tower's input

A genotype graph is built from HGVS. The gene hub node carries a learned
gene-identity embedding; each variant is its own node, with features read off
the HGVS notation, the catalog annotation and the allele dosage. Variants are
linked in genomic order.

Below, the same gene is built two ways — from star alleles and from an rsID —
to show that both routes end in HGVS.

In [ ]:
%%pgen
import polars as pl

from src.data.genotype import load_hgvs_catalog
from src.data.genotype.graph import GenotypeGraphBuilder, build_gene_vocab

catalog = load_hgvs_catalog()
rows = pl.read_parquet(get_settings().paths.library / "kg" / "training_rows.parquet")
vocab = build_gene_vocab(rows["gene"].to_list())
builder = GenotypeGraphBuilder(catalog, vocab)

# A CYP2C19 diplotype: *2 (no function) over *17 (increased function).
geno = builder.build("CYP2C19", "CYP2C19*2|CYP2C19*17", "*2/*17")
print("CYP2C19 *2/*17")
print(" ", geno)
print(f"  {geno.x.shape[0]} nodes (1 gene hub + {len(geno.hgvs)} variants), {geno.edge_index.shape[1]} edges")
print(f"  gene token: {int(geno.gene_token[0])} on the hub, 0 on variant nodes")
print(f"  function vector: {geno.geno_function.tolist()[0]}")
for h in geno.hgvs:
    info = catalog.info(h)
    print(f"    {h}  [{', '.join(info.so_terms) if info.so_terms else '-'}]")

# The same tower input, reached from an rsID and a called genotype instead.
other = builder.build("ABCB1", "rs1045642", "AG")
print("\nABCB1 rs1045642, genotype AG")
print(" ", other)
print("  ->", other.hgvs)

### A genotype graph, drawn

In [ ]:
%%pgen
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (graph_data, title) in zip(
    axes,
    [(geno, "CYP2C19 *2/*17"), (builder.build("CYP2D6", "CYP2D6*4|CYP2D6*1", "*1/*4"), "CYP2D6 *1/*4")],
):
    nodes = graph_data.x.shape[0]
    graph = nx.Graph()
    graph.add_nodes_from(range(nodes))
    graph.add_edges_from(graph_data.edge_index.t().tolist())

    # Node 0 is the gene hub; the rest are HGVS variants.
    labels = {0: title.split()[0]}
    labels.update({i + 1: h.split("g.")[-1] for i, h in enumerate(graph_data.hgvs)})

    nx.draw(
        graph,
        pos=nx.spring_layout(graph, seed=1),
        ax=ax,
        node_color=["#d95f02"] + ["#7570b3"] * (nodes - 1),
        node_size=[1500] + [900] * (nodes - 1),
        labels=labels,
        font_size=7,
        font_color="white",
        edge_color="#999999",
        width=1.5,
    )
    ax.set_title(f"{title}\n{nodes} nodes, {graph_data.edge_index.shape[1]} edges")

plt.tight_layout()
plt.show()

print("Orange = gene hub (learned gene-identity embedding)")
print("Purple = one HGVS variant each; edges are membership and cis-adjacency")

### Inside a variant node: the 45-feature HGVS vector and 3-feature edges

Every node in a genotype graph has width `HGVS_NODE_DIM = 45`, structured into 8 interpretable blocks (`src/data/genotype/features.py`), and every directed edge carries 3 features (`[is_membership, is_adjacency, log10_distance_norm]`):

| Columns | Width | Block | Source |
|---|---:|---|---|
| `0..1` | 2 | `node_type` | `[is_gene, is_variant]` (`[1, 0]` on hub node 0; `[0, 1]` on variant nodes) |
| `2..9` | 8 | `edit_kind` | One-hot HGVS edit (`substitution`, `deletion`, `duplication`, `insertion`, `delins`, `inversion`, `repeat`, `other`) |
| `10..12` | 3 | `level` | Molecular coordinate level (`genomic`, `transcript`, `protein`) |
| `13..17` | 5 | `region` | Coding region (`cds`, `intron`, `utr5`, `utr3`, `unknown`) |
| `18..21` | 4 | `seq_change` | `[transition, transversion, log_length, frameshift_like]` |
| `22..34` | 13 | `consequence` | 11-way Sequence Ontology multi-hot (`stop_gained`, `frameshift`, `splice`, ..., `missense`, ...) + `max_severity` + `consequence_known` (`src/data/genotype/consequence.py`) |
| `35..42` | 8 | `protein_change` | First-principles HGVS `p.` physicochemistry (`grantham_norm`, `abs_charge_delta`, `abs_hydropathy_delta`, `abs_volume_delta`, `abs_polarity_delta`, `is_stop_gain`, `is_frameshift`, `hgvs_protein_known` in `src/data/genotype/protein_change.py`) |
| `43..44` | 2 | `dosage` | `[alt_copies / 2.0, copies_known]` from diploid allele calls (`*2/*3`, `AG`, etc.) |

In [ ]:
%%pgen
from src.data.genotype.consequence import _GROUPS as SO_GROUPS
from src.data.genotype.features import EDIT_KINDS, HGVS_NODE_DIM, LEVELS, NODE_TYPES, REGIONS

FEATURE_NAMES = (
    [f"type:{t}" for t in NODE_TYPES]
    + [f"edit:{k}" for k in EDIT_KINDS]
    + [f"lvl:{l}" for l in LEVELS]
    + [f"reg:{r}" for r in REGIONS]
    + ["chg:transition", "chg:transversion", "chg:log_len", "chg:frameshift_like"]
    + [f"so:{g}" for g in SO_GROUPS]
    + ["so:max_severity", "so:known"]
    + [
        "p:grantham",
        "p:charge_delta",
        "p:hydropathy_delta",
        "p:volume_delta",
        "p:polarity_delta",
        "p:stop_gain",
        "p:frameshift",
        "p:known",
    ]
    + ["dosage:copies/2", "dosage:known"]
)
assert len(FEATURE_NAMES) == HGVS_NODE_DIM == dims.geno_features == 45

# Build CYP2C9 *2/*3 (two missense variants: radical p.Arg144Cys vs conservative p.Ile359Leu)
# and ABCB1 rs1045642 AG (diploid nucleotide call with explicit allele dosage).
cyp2c9 = builder.build("CYP2C9", "*2|*3", "*2/*3")
abcb1 = builder.build("ABCB1", "rs1045642", "AG")

print("=== CYP2C9 *2/*3 node feature breakdown ===")
for idx in range(cyp2c9.x.shape[0]):
    vec = cyp2c9.x[idx]
    if idx == 0:
        label = "Node 0 (Gene Hub: CYP2C9)"
    else:
        h = cyp2c9.hgvs[idx - 1]
        info = catalog.info(h)
        label = f"Node {idx} ({h} | {info.p_hgvs})"
    active = [(FEATURE_NAMES[j], float(vec[j])) for j in range(len(vec)) if abs(float(vec[j])) > 1e-4]
    print(f"  {label}")
    print("   ", ",  ".join(f"{k}={v:.2f}" for k, v in active))

print("\n=== CYP2C9 *2/*3 directed edges (u -> v : [is_membership, is_adjacency, log10_distance_norm]) ===")
for (u, v), attr in zip(cyp2c9.edge_index.t().tolist(), cyp2c9.edge_attr.tolist()):
    bp_str = f" (~{int(round(10 ** (attr[2] * 7.0) - 1)):,} bp apart)" if attr[1] > 0.5 else " (hub <-> variant)"
    print(f"  {u} -> {v}   [{attr[0]:.0f}, {attr[1]:.0f}, {attr[2]:.3f}]{bp_str}")

# Heatmap of the 45 features across hub + CYP2C9*2 (p.Arg144Cys) + CYP2C9*3 (p.Ile359Leu) + ABCB1 rs1045642
mat = torch.vstack([cyp2c9.x, abcb1.x[1:]]).numpy()
row_labels = [
    "CYP2C9 Hub (node 0)",
    "CYP2C9*2 (p.Arg144Cys)",
    "CYP2C9*3 (p.Ile359Leu)",
    "ABCB1 rs1045642 (AG)",
]

fig, ax = plt.subplots(figsize=(15, 3.8))
im = ax.imshow(mat, aspect="auto", cmap="YlGnBu", vmin=0.0, vmax=1.0)
ax.set_yticks(range(len(row_labels)), row_labels, fontsize=9)
ax.set_xticks(range(len(FEATURE_NAMES)), FEATURE_NAMES, rotation=90, fontsize=7.5)

block_boundaries = [2, 10, 13, 18, 22, 35, 43]
block_labels = [
    (0.5, "type(2)"), (5.5, "edit_kind(8)"), (11.0, "lvl(3)"), (15.0, "region(5)"),
    (19.5, "seq_chg(4)"), (28.0, "SO consequence(13)"), (38.5, "protein_change(8)"), (43.5, "dosage(2)"),
]
for b in block_boundaries:
    ax.axvline(b - 0.5, color="#d95f02", linewidth=1.3, linestyle="--")
for xpos, bname in block_labels:
    ax.text(xpos, -0.65, bname, ha="center", va="bottom", fontsize=7.5, fontweight="bold")

plt.colorbar(im, ax=ax, fraction=0.015, pad=0.01, label="feature value [0, 1]")
ax.set_title("45-dimensional HGVS genotype node features across hub and variant nodes", pad=22)
plt.tight_layout()
plt.show()

### Corpus-wide genotype resolution and allele dosage coverage

Running `GenotypeGraphBuilder` across all **26,544 training rows** shows how often each identifier type (direct HGVS, star allele, rsID, or annotation fallback) resolves through `hgvs_catalog.pt` (`f8572592acb7...`) and how many rows carry explicit diploid allele calls (`AlleleCall.parse`).

In [ ]:
%%pgen
from src.data.genotype.graph import AlleleCall

corpus_builder = GenotypeGraphBuilder(catalog, vocab)
dosage_parsed = 0
node_counts = []

for gene, gt, al, ann in zip(
    rows["gene"].to_list(),
    rows["genotype"].to_list(),
    rows["alleles"].to_list(),
    rows["variant_annotation_id"].to_list(),
):
    g = corpus_builder.build(gene, gt, al, annotation_id=ann)
    node_counts.append(int(g.x.shape[0]))
    if AlleleCall.parse(al) is not None:
        dosage_parsed += 1

st = corpus_builder.stats
n_total = st["rows"]
n_with_var = st["rows_with_variants"]
n_hub_only = n_total - n_with_var
print(f"Total training rows processed : {n_total:,}")
print(f"Rows resolved to >=1 HGVS     : {n_with_var:,} ({100 * n_with_var / n_total:.1f}%)")
print(f"Rows with hub-only (e.g. *1)  : {n_hub_only:,} ({100 * n_hub_only / n_total:.1f}%)")
print(f"Rows with diploid AlleleCall  : {dosage_parsed:,} ({100 * dosage_parsed / n_total:.1f}%)")
print(f"Unparseable HGVS tokens       : {st['unparseable_hgvs']}")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))

sources = {
    "Direct HGVS": st["variants_from_hgvs"],
    "Annotation IMPLICATES": st["variants_from_annotation"],
    "Star allele (*N)": st["variants_from_star"],
    "dbSNP rsID": st["variants_from_rsid"],
}
axes[0].barh(list(sources.keys())[::-1], list(sources.values())[::-1], color="#7570b3")
axes[0].set_title("Variant links by resolution path")
axes[0].set_xlabel("variant links across 26,544 rows")
for i, v in enumerate(list(sources.values())[::-1]):
    axes[0].text(v + 300, i, f"{v:,}", va="center", fontsize=8.5)
axes[0].set_xlim(0, max(sources.values()) * 1.18)

vals, cnts = np.unique(node_counts, return_counts=True)
axes[1].bar(vals[:12], cnts[:12], color="#d95f02")
axes[1].set_title("Genotype graph size (1 hub + N variants)")
axes[1].set_xlabel("nodes per genotype graph")
axes[1].set_ylabel("training rows")
axes[1].set_xticks(vals[:12])

plt.tight_layout()
plt.show()

---

## Drug sets: combinations and classes

A training row's drug field is not always one molecule. It holds `|`-joined
**entity keys**: `CHEMBL<n>` for a drug or a combination component, or
`ATC:<codes>` for a drug *class*, which pools over its ATC members.

`GraphCache.get_entity` returns every molecule behind one key; the collator then
tells the model which molecules belong to which entity, and `DrugSetPooling`
averages within an entity before attending across the row.

In [ ]:
%%pgen
for key in ("CHEMBL1771", "ATC:C01", "ATC:N02BE"):
    kind, molecules = cache.get_entity(key)
    names = []
    for m in molecules[:6]:
        hit = compounds.filter(pl.col("inchikey") == m.inchikey)
        names.append(hit["title"].item() if hit.height and hit["title"].item() else m.drug_id)
    suffix = ", ..." if len(molecules) > 6 else ""
    print(f"{key:14} kind={kind}  {len(molecules):3} molecule(s)")
    print(f"{'':14} {', '.join(names)}{suffix}\n")

Class rows are useful training signal — a PharmGKB annotation often concerns "a
statin" rather than one named statin — but they are a leakage hazard: a class
pool can contain the very drug being held out. The protocol keeps class rows in
**train only**, and in the `drug_cold` scenario the training cache's class pools
drop held-out drugs. Notebook `04` shows that machinery.

---

## A batch

What one forward pass actually receives: the collator packs variable-size graphs
into flat tensors plus index maps.

In [ ]:
%%pgen
from src.data.collator import DoubleTowerCollater
from src.data.datasets import DoubleTowerDataset

print("Row fields:", rows.columns, "\n")

# Pick 4 illustrative rows:
#   0: single drug (clopidogrel CHEMBL1771) + CYP2C19 HGVS
#   1: multi-drug combination (multiple ChEMBL IDs)
#   2: ATC class row (ATC:C10AA statins + SLCO1B1)
#   3: bare HGVS variant row
idx_single = int(rows.with_row_index().filter(
    (pl.col("drugs_chembl") == "CHEMBL1771") & (pl.col("gene") == "CYP2C19")
)["index"][0])
idx_combo = int(rows.with_row_index().filter(
    pl.col("drugs_chembl").str.contains(r"\|") & (~pl.col("is_class_row"))
)["index"][0])
idx_class = int(rows.with_row_index().filter(
    pl.col("drugs_chembl") == "ATC:C10AA"
)["index"][0])
idx_hgvs = int(rows.with_row_index().filter(
    pl.col("genotype").str.starts_with("NC_") & (~pl.col("is_class_row"))
)["index"][10])

demo_indices = [idx_single, idx_combo, idx_class, idx_hgvs]
print("4 representative rows selected for collation:")
print(rows[demo_indices].select("drugs_chembl", "drugs", "gene", "genotype", "alleles", "phenotype_category", "is_class_row"), "\n")

# Build dataset on a small slice (4 focal rows + 30 background rows so all target classes are present)
slice_indices = demo_indices + list(range(30))
rows_slice = rows[slice_indices]

dataset = DoubleTowerDataset(
    df=rows_slice,
    drug_col="drugs_chembl",
    geno_col="genotype",
    target_cols=["phenotype_category"],
    multilabel_cols={"phenotype_category"},
    gene_col="gene",
    genotype_builder=GenotypeGraphBuilder(catalog, vocab),
    drug_cache=cache,
    dims=dims,
    inference_mode=True,
)
collator = DoubleTowerCollater(inference_mode=True)
batch = collator([dataset[i] for i in range(4)])

db = batch["drug_batch"]
gb = batch["geno_batch"]
n_mols = int(db.batch.max().item() + 1)
n_ents = int(db.entity_kind.numel())
n_rows = int(db.entity_to_row.max().item() + 1)

print("=== Collated drug_batch ===")
print(f"  Batch rows          : {n_rows}")
print(f"  Entities in batch   : {n_ents}  (entity_to_row = {db.entity_to_row.tolist()}, entity_kind = {db.entity_kind.tolist()}  [0=drug, 1=class])")
print(f"  Molecules in batch  : {n_mols}  (mol_to_entity = {db.mol_to_entity.tolist()})")
print(f"  Atom tensor x       : {tuple(db.x.shape)}   | edge_index: {tuple(db.edge_index.shape)}   | edge_attr: {tuple(db.edge_attr.shape)}")
print(f"  Pharmacophore x     : {tuple(db.pharm_x.shape)}   | pharm_pos : {tuple(db.pharm_pos.shape)}   | pharm_x_batch: {tuple(db.pharm_x_batch.shape)}")
print(f"  Global & ADMET      : global_feats {tuple(db.global_feats.shape)}, admet_feats {tuple(db.admet_feats.shape)}")

print("\n=== Collated geno_batch ===")
print(f"  Total nodes         : {tuple(gb.x.shape)}  (batch assignment = {gb.batch.tolist()})")
print(f"  Edges               : {tuple(gb.edge_index.shape)}  | edge_attr: {tuple(gb.edge_attr.shape)}")
print(f"  Gene tokens         : {gb.gene_token.tolist()}")

classes = list(dataset.target_encoder.encoders["phenotype_category"].classes_)
print("\n=== Encoded multi-label targets ===")
for axis, tensor in batch["targets"].items():
    print(f"  {axis:20}: shape={tuple(tensor.shape)}, classes={classes}")
    print(tensor)

### Untrained forward pass through `PharmagenTwoTower`

Even before full GPU training on the rebuilt library, we can instantiate `PharmagenTwoTower` (`src/model/architectures/model.py`) with its 2D molecular tower, 3D pharmacophore tower (`SchNet` / `PharmacophoreGNN`), hierarchical `DrugSetPooling`, HGVS genotype tower, and bidirectional cross-attention fusion, and pass our 4-row real batch through it.

In [ ]:
%%pgen
from src.config import get_axes_config, get_model_config
from src.model.architectures.assembly import infer_axis_specs
from src.model.engine.base import build_gnn_model

model_cfg = get_model_config("TwoTowerGAT")
axis_specs = infer_axis_specs(
    dataset.target_encoder.encoders,
    dataset.targets,
    get_settings().multi_label_set,
    get_axes_config(),
)
model = build_gnn_model(
    dims=dims,
    drug_dim=dims.drug_features,
    geno_dim=dims.geno_features,
    axes=axis_specs,
    params=model_cfg.training_params(),
    device=torch.device("cpu"),
    switches={"use_pharmacophore": True, "use_cross_attention": True, "use_polypharmacy": False},
    gene_vocab=vocab,
)
model.eval()

param_counts = model.count_parameters()
print("PharmagenTwoTower parameter breakdown:")
for submodule, count in param_counts.items():
    print(f"  {submodule:22} {count:>10,}")

with torch.no_grad():
    outputs = model(batch["drug_batch"], batch["geno_batch"], return_attention=True)

logits = outputs["phenotype_category"]
probs = torch.sigmoid(logits)
print(f"\nOutput logits ['phenotype_category']: shape {tuple(logits.shape)}")
print("Untrained sigmoid probabilities across the 4 rows:")
for row_i, idx in enumerate(demo_indices):
    row_info = rows.row(idx, named=True)
    p_str = ", ".join(f"{cls}={probs[row_i, j]:.3f}" for j, cls in enumerate(classes))
    print(f"  Row {row_i} ({row_info['drugs_chembl'][:18]:18} + {row_info['gene']:8}): {p_str}")

attn = outputs["_attention"]
print(f"\nCross-attention weight tensor outputs['_attention']: shape={tuple(attn.shape)}")
print("Row 0 attention weights (drug entities x genotype nodes):", attn[0].tolist())

Those three rows happen to show all three genotype notations the sources use —
star alleles (`CYP3A5*1|CYP3A5*3`), an rsID (`rs1799752`) and bare HGVS
(`NC_000007.14:g.87509329A>G`). All three resolve through the catalog to the
same kind of graph, which is what "the genotype tower is keyed by HGVS" buys.

The dataset and collator are exercised end to end by
`tests/integration/test_train_predict_e2e.py`, which trains for a couple of
steps on synthetic graphs and runs a prediction — the wiring shown above, under
test.